In [36]:
import os
from pathlib import Path
import sys

# PDF libraries
from pypdf import PdfReader

# Docling
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, RapidOcrOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

# Regex
import re

# OpenRouter
from openrouter import OpenRouter
from openrouter.operations import CreateEmbeddingsResponse

# env
from dotenv import load_dotenv

# Calculate token size
from pytokencalc import count_tokens

# Hash
import hashlib

# Import from project files
sys.path.append('..')

from config import open_router_client
from config import supabase_client


# 0 - define the sample docs path

In [37]:
if (Path(os.getcwd()) != Path("c:/Users/alexi/studcess/api")):
    os.chdir("../../../api")

print(os.getcwd())

docs = Path("sample-data")
sample = docs / "exam1-2022-sol-1.pdf"

c:\Users\alexi\studcess\api


# 1 - extract document metadata

In [38]:
def extract_metadata(pdf_path: Path)->None:
    reader = PdfReader(pdf_path)
    metadata = reader.metadata
    num_pages = len(reader.pages)
    
    print(metadata)
    print(num_pages)
    
    # Build custom metadata
    info = {num_pages:num_pages}
    
    # Regexes for file class 
    EXAM_KEYWORDS = ["Exam"]
    
extract_metadata(sample)

{'/Producer': 'pdfTeX-1.40.20', '/Author': "Dr Pierre-André Mudry, Filière ISC - Haute Ecole d'Ingénierie, Sion", '/Title': 'Test intermédiaire', '/Subject': '', '/Creator': 'LaTeX with hyperref', '/Keywords': '101 - Fondements de la programmation 1, exercises', '/CreationDate': "D:20221109160320+01'00'", '/ModDate': "D:20221109160320+01'00'", '/Trapped': '/False', '/PTEX.Fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.20 (TeX Live 2019/Debian) kpathsea version 6.3.1'}
12


# 2 - convert document to markdown with docling

In [39]:
%%time

pipeline_options = PdfPipelineOptions(
    do_ocr=True,
    do_table_structure=True,
    generate_page_images=False,
    do_picture_description=False,
    ocr_options=RapidOcrOptions(),
    #picture_description_options=smolvlm_picture_description
)

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options),
    }
)

PAGE_BREAK_PLACEHOLDER = "<!-- page_break -->"

def process_document(pdf_path: Path,converter: DocumentConverter, n_pages=-1)->str:
    result = converter.convert(pdf_path)
    doc = result.document
    print(f"\nNumber of pages: {len(doc.pages)}\n")

    converted_dir = docs / "converted"
    converted_dir.mkdir(parents=True, exist_ok=True)
    
    doc.save_as_markdown(converted_dir / f"{pdf_path.stem}.md")
    doc.save_as_json(converted_dir / f"{pdf_path.stem}.json")
    
    markdown = doc.export_to_markdown(page_break_placeholder=PAGE_BREAK_PLACEHOLDER)
    
    if n_pages == -1:
        return markdown
    
    return PAGE_BREAK_PLACEHOLDER.join(markdown.split(PAGE_BREAK_PLACEHOLDER)[:n_pages])
    
doc = process_document(sample,converter,2)
print(doc)

Loading weights: 100%|██████████| 770/770 [00:00<00:00, 3984.83it/s]



Number of pages: 12

Prénom : . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .

<!-- image -->

## TEST INTERMÉDIAIRE SOLUTION 101.1 - Programmation impérative

## Consigne :

Lisez attentivement la donnée et répondez de manière lisible aux questions. Vous avez droit pour cet examen à un aide-mémoire de 1 pages (1 feuille recto). Aucun moyen électronique n'est permis.

Un conseil : ne restez pas bloqués sur une question. Répondez tout d'abord aux questions avec lesquelles vous êtes à l'aise et revenez ensuite aux questions posant problème. Le barème indiqué est indicatif.

| Question              |   Points |   Bonus | Score   |
|-----------------------|----------|---------|---------|
| Short questions       |       10 |       2 |         |
| Loops analysis        |        8 |       0 |         |
| EBNF grammars         |        6 |       0 |         |
| Code comprehension    |        3 |       0 |         |
| Writing functions     |        7 |       1 |         |
| S

# 3 - Implement chunking 

We do a first pass by splitting by heading as this will be where we are most likely to get relevant information

In [40]:
HEADING_PATTERN = "\n#"

raw_chunks = doc.split(HEADING_PATTERN)

chunked_text = ""
for i,chunk in enumerate(raw_chunks):
    if chunk.startswith("#"):
        chunk = f"#{chunk}" # Put back the header
    chunked_text += f"<start_chunk_{i}>\n{chunk}<end_chunk_{i}>"

print(chunked_text)

<start_chunk_0>
Prénom : . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .

<!-- image -->
<end_chunk_0><start_chunk_1>
## TEST INTERMÉDIAIRE SOLUTION 101.1 - Programmation impérative
<end_chunk_1><start_chunk_2>
## Consigne :

Lisez attentivement la donnée et répondez de manière lisible aux questions. Vous avez droit pour cet examen à un aide-mémoire de 1 pages (1 feuille recto). Aucun moyen électronique n'est permis.

Un conseil : ne restez pas bloqués sur une question. Répondez tout d'abord aux questions avec lesquelles vous êtes à l'aise et revenez ensuite aux questions posant problème. Le barème indiqué est indicatif.

| Question              |   Points |   Bonus | Score   |
|-----------------------|----------|---------|---------|
| Short questions       |       10 |       2 |         |
| Loops analysis        |        8 |       0 |         |
| EBNF grammars         |        6 |       0 |         |
| Code comprehension    |        3 |       0 |         |
| Writing 

# 4 - Embed chunks

## Golden rule for chunk sizes
We divide the length of text by 4 to get the approximate token count:
- Between 100 and 1000 => acceptable
- Less than 100 => too low, merge with another chunk or check for junk and discard
- More than 1000 => too big, split into smaller chunks needed

In [41]:
EMBEDDING_MODEL = "openai/text-embedding-3-small"

class Chunk:
    def __init__(self,embedding:list[float],text:str)->None:
        self.embedding = embedding
        self.text = text
        self.content_hash = hashlib.sha256(text.encode("utf-8")).hexdigest()
        self.row = {
            "content": self.text,
            "content_hash": self.content_hash,
            "embedding": self.embedding
        }
    
    def __repr__(self)->str:
        return f"{self.row}"

def embed_texts(raw_chunks: list[str], embedding_model: str) -> list[Chunk]:
    chunks: list[Chunk] = []

    for text in raw_chunks:
        tokens_count = len(text) // 4
        print(f"tokens count {tokens_count}")

        try:
            res = open_router_client.embeddings.generate(
                input=text,
                model=embedding_model,
            )
            print("Chunk successfully embedded")
        except Exception as e:
            print(f"Error embedding chunk: {e}")

        embedding = res.data[0].embedding   # list of lists of floats
        chunks.append(Chunk(embedding, text))

    return chunks

chunks = embed_texts(raw_chunks,EMBEDDING_MODEL)

tokens count 22
Chunk successfully embedded
tokens count 15
Chunk successfully embedded
tokens count 268
Chunk successfully embedded
tokens count 382
Chunk successfully embedded


# Store chunks inside the database

In [42]:
def store_chunks(chunks:list[Chunk]):
    for chunk in chunks:
        try:
            # Only insert new row if the content is unique, otherwise update
            response = supabase_client.table("chunks").upsert(chunk.row, on_conflict="content_hash").execute()
            print(f"Successfully inserted {len(response.data)} chunks.")
        except Exception as e:
            print(f"Error inserting data: {e}")

store_chunks(chunks)

Successfully inserted 1 chunks.
Successfully inserted 1 chunks.
Successfully inserted 1 chunks.
Successfully inserted 1 chunks.


# Retrieval

In [ ]:
def retrieve(question: str, k: int = 3):
    q_vec = embed_texts([question],EMBEDDING_MODEL)[0]  
     
    # same model as the chunks
    try: 
        res = supabase_client.rpc("match_chunks", {
            "query_embedding": q_vec.embedding,
            "match_count": k,
        }).execute()
        print("Question embedded successfully")
        print(res.data)
    except Exception as e:
        print(f"Error embedding question: {e}")
        
    return res.data
    
retrieve("What is the solution to (b) inside Question 1?")

tokens count 11
Chunk successfully embedded
Question embedded successfully
[{'id': 'f1d262a3-ce7d-4f67-a1d7-aaf4627e8352', 'content': '# Question 1 - Short questions (10 points)\n\nCette question est séparée en plusieurs exercices indépendants. Le nombre de point pour chaque exercice est indiqué dans la marge.\n\n- (a) [4 Pt] Soient les déclarations suivantes :\n\n```\n1 val foo : Int = 0xBABA 2 val foobar : Float = 3.825f 3 val baz : String = "y"\n```\n\nLes expressions suivantes sont correctes. Donnez le type et la valeur des expressions suivantes.\n\n- 1) (foobar + - foobar).toByte\n- 2) (foo &amp; 0xFF0).toHexString\n- 3) (((foo &gt;&gt; 4) &lt;&lt; 4) | 0xE).toHexString\n- 4) (foo^foo).toShort\n- 5) ((foobar * 100).toInt / 100.0f)\n- 6) if(foo &lt; 0xFFFF) \'a\' else \'b\'\n- 7) if(true) \'1\' + baz else baz + \'1\'\n- 8) baz + (\'d\' + 1.9).toChar + (\'t\'-1).toChar\n9. (b) [1 Pt] Quel est le contenu de r après l\'exécution du code ci-dessous :\n\n```\n1 val s: String = "TikTok" 

[{'id': 'f1d262a3-ce7d-4f67-a1d7-aaf4627e8352',
  'content': '# Question 1 - Short questions (10 points)\n\nCette question est séparée en plusieurs exercices indépendants. Le nombre de point pour chaque exercice est indiqué dans la marge.\n\n- (a) [4 Pt] Soient les déclarations suivantes :\n\n```\n1 val foo : Int = 0xBABA 2 val foobar : Float = 3.825f 3 val baz : String = "y"\n```\n\nLes expressions suivantes sont correctes. Donnez le type et la valeur des expressions suivantes.\n\n- 1) (foobar + - foobar).toByte\n- 2) (foo &amp; 0xFF0).toHexString\n- 3) (((foo &gt;&gt; 4) &lt;&lt; 4) | 0xE).toHexString\n- 4) (foo^foo).toShort\n- 5) ((foobar * 100).toInt / 100.0f)\n- 6) if(foo &lt; 0xFFFF) \'a\' else \'b\'\n- 7) if(true) \'1\' + baz else baz + \'1\'\n- 8) baz + (\'d\' + 1.9).toChar + (\'t\'-1).toChar\n9. (b) [1 Pt] Quel est le contenu de r après l\'exécution du code ci-dessous :\n\n```\n1 val s: String = "TikTok" 2 val r: String = StringUtils.charAt(s, StringUtils.length(s)-2) + 3 "" +

# Ask the questions

In [54]:
RAG_PROMPT = """
You are a computer science professor and you are grading an exam about the Scala fundamentals. 

Use the context:
{context}

To answer the question:
{question}

Give me only the solution on the document, dont solve yourself and answer directly
""".strip()


In [55]:
LLM = "openai/gpt-4o-mini"

def prompt(question: str)->str:
    chunks = retrieve(question, k=1)
    
    context = ""
    for i,chunk in enumerate(chunks):
        context += f"<Chunk {i}>"
        context += chunk['content']
        context += f"<Chunk {i}>"
    
    rag_prompt = RAG_PROMPT.format(context=context,question=question)
    
    try:
        response = open_router_client.chat.send(
            model=LLM,
            messages=[{
                "content": f"{rag_prompt}",
                "role": "system"
            }]
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error prompting model: {e}")
        
answer = prompt("What is the solution to (b) inside Question 1?")
print(answer)

tokens count 11
Chunk successfully embedded
Question embedded successfully
[{'id': 'f1d262a3-ce7d-4f67-a1d7-aaf4627e8352', 'content': '# Question 1 - Short questions (10 points)\n\nCette question est séparée en plusieurs exercices indépendants. Le nombre de point pour chaque exercice est indiqué dans la marge.\n\n- (a) [4 Pt] Soient les déclarations suivantes :\n\n```\n1 val foo : Int = 0xBABA 2 val foobar : Float = 3.825f 3 val baz : String = "y"\n```\n\nLes expressions suivantes sont correctes. Donnez le type et la valeur des expressions suivantes.\n\n- 1) (foobar + - foobar).toByte\n- 2) (foo &amp; 0xFF0).toHexString\n- 3) (((foo &gt;&gt; 4) &lt;&lt; 4) | 0xE).toHexString\n- 4) (foo^foo).toShort\n- 5) ((foobar * 100).toInt / 100.0f)\n- 6) if(foo &lt; 0xFFFF) \'a\' else \'b\'\n- 7) if(true) \'1\' + baz else baz + \'1\'\n- 8) baz + (\'d\' + 1.9).toChar + (\'t\'-1).toChar\n9. (b) [1 Pt] Quel est le contenu de r après l\'exécution du code ci-dessous :\n\n```\n1 val s: String = "TikTok" 